In [1]:
import os
import re
import h5py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from scipy.io import loadmat
from scipy.signal import stft, welch
from tqdm import tqdm

This script must be placed at the same directory level as the
Preprocessed_EEG folder, which is located inside the SEED_EEG
folder obtained from the official SEED dataset download.

Expected directory structure:

    SEED_EEG/
    ├── Preprocessed_EEG/
    │   ├── label.mat
    │   ├── 1_20131027.mat
    │   ├── 1_20131030.mat
    │   └── ...
    └── this_script.py

The SEED dataset can be downloaded from the official BCMI website:
https://bcmi.sjtu.edu.cn/home/seed/seed.html

In [2]:
# folder containing the raw preprocessed .mat files from the SEED dataset
folder = "Preprocessed_EEG"

# output HDF5 file that will consolidate all EEG trials and metadata
preprocessed_h5 = "preprocessed_EEG.h5"

# output HDF5 file where all EEG trials are zero-padded to uniform length
padded_h5 = "padded_EEG.h5"

# list all .mat files in the folder, excluding the label file
files = [
    f for f in os.listdir(folder) 
    if f.endswith(".mat") 
    and f != "label.mat"]

In [3]:
# --- extract patient and date from filenames ---
# expected filename format: <patient_id>_<YYYYMMDD>.mat (e.g. 3_20130522.mat)
file_info = {}
for file_name in files:
    match = re.match(r"(\d+)_(\d{8})\.mat", file_name)
    patient, date = match.groups()
    file_info[file_name] = {
        "patient": int(patient),
        "experiment": int(date), # stored as integer date for sorting
    }

# --- convert experiment date to sequential session index per patient ---
# each patient may have multiple recording sessions on different days
# sessions are sorted chronologically and indexed starting from 1
for p in set(v["patient"] for v in file_info.values()):
    patient_files = [(file_name, int(v["experiment"])) for file_name, v in file_info.items() if v["patient"] == p]
    patient_files.sort(key=lambda x: x[1]) # sort by date to ensure chronological order
    for exp_idx, (file_name, _) in enumerate(patient_files, start=1):
        file_info[file_name]["experiment"] = exp_idx # replace date with session index

# --- create HDF5 file and write all EEG trials ---
# each trial is stored as a separate dataset named P<patient>_E<session>_T<trial>
with h5py.File(preprocessed_h5, "w") as h5f:
    for file_name, info in tqdm(file_info.items(), desc="Files", unit="file"):
        patient = info["patient"]
        experiment = info["experiment"]
        file_path = os.path.join(folder, file_name)

        # load .mat file; simplify_cells=True flattens nested MATLAB structs
        data = loadmat(file_path, simplify_cells=True)

        # find all EEG trial keys (e.g. djc_eeg1, djc_eeg2, ...)
        eeg_keys = [k for k in data.keys() if "_eeg" in k]
        for key in eeg_keys:
            # extract trial number from the end of the key name
            trial_match = re.search(r"(\d+)$", key)
            trial = int(trial_match.group(1))

            # build dataset name: P03_E02_T05
            dataset_name = f"P{patient:02}_E{experiment:02}_T{trial:02}"

            # write trial EEG data as float32 to save disk space
            h5f.create_dataset(dataset_name, data=np.array(data[key], dtype=np.float32))

    # --- load and process emotion labels from label.mat ---
    label_mat = loadmat(os.path.join(folder, "label.mat"), simplify_cells=True)

    # get the first non-private key (the actual label array)
    label_keys = [k for k in label_mat.keys() if not k.startswith("__")]
    labels = np.array(label_mat[label_keys[0]], dtype=np.int16).flatten()

    # shift labels from [-1, 0, 1] to [0, 1, 2] for PyTorch compatibility
    # -1 (negative) -> 0, 0 (neutral) -> 1, 1 (positive) -> 2
    labels += 1
    
    # --- replicate labels to match the total number of trials ---
    n_trials = len([k for k in h5f.keys() if "_T" in k])
    n_stacks = n_trials // len(labels)
    labels_full = np.tile(labels, n_stacks)
    h5f.create_dataset("labels", data=labels_full)

    # --- store metadata ---
    h5f.create_dataset("n_classes", data=np.array([3], dtype=np.int16))
    h5f.create_dataset("sample_rate", data=np.array([200], dtype=np.int16))

print(f"\nHDF5 file created: {preprocessed_h5}")

Files: 100%|██████████████████████████████████| 45/45 [02:19<00:00,  3.10s/file]


HDF5 file created: preprocessed_EEG.h5


In [4]:
# --- inspect the resulting HDF5 file ---
with h5py.File(preprocessed_h5 , "r") as h5f:
    print(f"Keys in HDF5 file ({len(h5f.keys())} datasets):\n")
    for key in h5f.keys():
        dataset = h5f[key]
        print(f"{key}: shape={dataset.shape}, dtype={dataset.dtype}")

Keys in HDF5 file (678 datasets):

P01_E01_T01: shape=(62, 47001), dtype=float32
P01_E01_T02: shape=(62, 46601), dtype=float32
P01_E01_T03: shape=(62, 41201), dtype=float32
P01_E01_T04: shape=(62, 47601), dtype=float32
P01_E01_T05: shape=(62, 37001), dtype=float32
P01_E01_T06: shape=(62, 39001), dtype=float32
P01_E01_T07: shape=(62, 47401), dtype=float32
P01_E01_T08: shape=(62, 43201), dtype=float32
P01_E01_T09: shape=(62, 53001), dtype=float32
P01_E01_T10: shape=(62, 47401), dtype=float32
P01_E01_T11: shape=(62, 47001), dtype=float32
P01_E01_T12: shape=(62, 46601), dtype=float32
P01_E01_T13: shape=(62, 47001), dtype=float32
P01_E01_T14: shape=(62, 47601), dtype=float32
P01_E01_T15: shape=(62, 41201), dtype=float32
P01_E02_T01: shape=(62, 47001), dtype=float32
P01_E02_T02: shape=(62, 46601), dtype=float32
P01_E02_T03: shape=(62, 41201), dtype=float32
P01_E02_T04: shape=(62, 47601), dtype=float32
P01_E02_T05: shape=(62, 37001), dtype=float32
P01_E02_T06: shape=(62, 39001), dtype=float32

In [5]:
# target number of samples per channel: 4 minutes at 200 Hz = 48000 samples
n_samples_pad = 48000

# --- create a new HDF5 where all EEG trials have uniform length ---
# trials shorter than 48000 samples are zero-padded at the end
# trials longer than 48000 samples are truncated
with h5py.File(preprocessed_h5, "r") as h5:
    eeg_keys = [k for k in h5.keys() if "_T" in k]
    n_datasets = len(eeg_keys)

    # preserve metadata from the original file
    labels = h5["labels"][()]
    n_classes = h5["n_classes"][()]
    sample_rate = h5["sample_rate"][()]

    # infer number of EEG channels from the first trial
    first_data = h5[eeg_keys[0]][()]
    n_channels = first_data.shape[0]

    # create output HDF5 with pre-allocated arrays for efficient writing
    with h5py.File(padded_h5, "w") as pad_h5:
        # main EEG array: (n_trials, n_channels, n_samples)
        data_dset = pad_h5.create_dataset("data", shape=(n_datasets, n_channels, n_samples_pad), dtype='float32')


        # string dataset to track the source trial name for each row
        dt = h5py.string_dtype(encoding='utf-8')
        source_dset = pad_h5.create_dataset(
            "source", shape=(n_datasets,), dtype=dt
        )

        for i, key in enumerate(tqdm(eeg_keys, desc="Processing EEG", unit="dataset")):
            data = h5[key][()]  # shape: (n_channels, n_samples)
            n_ch, n_samples = data.shape

            # pad with zeros if signal is shorter than target length
            if n_samples < n_samples_pad:
                data = np.pad(data, ((0,0),(0,n_samples_pad-n_samples)), mode='constant') # zero-padding on the time axis

            # truncate if signal is longer than target length
            elif n_samples > n_samples_pad:
                data = data[:, :n_samples_pad]

            # write trial to pre-allocated dataset
            data_dset[i] = data
            source_dset[i] = key # store original trial name for traceability

        # copy metadata to the new file
        pad_h5.create_dataset("labels", data=labels)
        pad_h5.create_dataset("n_classes", data=n_classes)
        pad_h5.create_dataset("sample_rate", data=sample_rate)

print(f"New padded HDF5 file created: {padded_h5}")

Processing EEG: 100%|████████████████████| 675/675 [01:56<00:00,  5.80dataset/s]

New padded HDF5 file created: padded_EEG.h5


In [6]:
# --- inspect the padded HDF5 file ---

with h5py.File(padded_h5, "r") as h5f:
    print(f"Keys in HDF5 file ({len(h5f.keys())} datasets):\n")
    for key in h5f.keys():
        dataset = h5f[key]
        print(f"{key}: shape={dataset.shape}, dtype={dataset.dtype}")

Keys in HDF5 file (5 datasets):

data: shape=(675, 62, 48000), dtype=float32
labels: shape=(675,), dtype=int16
n_classes: shape=(1,), dtype=int16
sample_rate: shape=(1,), dtype=int16
source: shape=(675,), dtype=object
